# Datasets preprocessing

In [2]:
#release; stable

from src.utils import myrepr
import json
import os
import argparse
import subprocess
import shlex

# Function to create and submit SLURM job scripts
def submit_slurm_job(script_path, args, folder, job_name, submit):
    # Define SLURM job parameters
    
    outs_folder = os.path.join(folder, "slurm_outs")
    sh_folder = os.path.join(folder, "slurm_sh")
    os.makedirs(outs_folder, exist_ok=True)
    os.makedirs(sh_folder, exist_ok=True)
    slurm_script = os.path.join(sh_folder, f"{job_name}.sh")
    
    
    # Create SLURM job script
    with open(slurm_script, "w") as f:
        f.write("#!/bin/bash\n")
        f.write(f"#SBATCH --job-name={job_name}\n")
        f.write("#SBATCH --cpus-per-task=1\n")
        f.write("#SBATCH -N 1\n")
        f.write("#SBATCH --partition=batch\n")
        f.write("#SBATCH --mem=128GB\n")
        f.write("#SBATCH --time=6:00:00\n")
        f.write(f"#SBATCH --output={outs_folder}/{job_name}.out\n")
        f.write(f"python {script_path} ")
        for arg in args:
            f.write(f"{arg} ")
        f.write("\n")

    # Submit the job using sbatch and capture the output
    if submit:
        result = subprocess.run(["sbatch", slurm_script], capture_output=True, text=True)
        
        # Print the result of the job submission
        if result.returncode == 0:
            print(f"Job submitted successfully: {result.stdout.strip()}")
        else:
            print(f"Job submission failed: {result.stderr.strip()}")

# Define the JSON configuration directly within the script
json_config = """
{
    "version": "0.2.0",
    "configurations": [
        {
            "name": "data_preprocessing",
            "type": "debugpy",
            "request": "launch",
            "program": "/ibex/user/sokoi0a/_projects/2024_p-LORA-experiments/LinReg/data_preprocessing.py",
            "console": "integratedTerminal",
            "justMyCode": true,
            "args": [
                "--dataset", "synthetic_dense",
                "--loss_func", "lin-reg",
                "--regularizer_type", "non-cvx",
                "--use_ray", "0",
                "--is_sparse_dataset", "0",
                "--generate_dataset", "1",
                "--is_minimize", "0",
                "--load_prepared_dataset", "0",
                "--load_raw_dataset", "0",
                "--print_args", "1",
                "--print_status", "1",
                "--hetero", "0",
                "--num_workers", "1",
                "--num_samples", "100000",
                "--dim", "4096",
                "--computable_params", "[]",
                "--loadable_params", "[]"
            ],
            "python": "/home/sokoi0a/anaconda3/envs/pyten/bin/python"
        }
    ]
}
"""

submit = 1

# Parse the JSON configuration
config = json.loads(json_config)
# Extract the script path and args
script_path = config["configurations"][0]["program"]
args = config["configurations"][0]["args"]
# Properly format the --computable_params argument
for i in range(len(args)):
    if args[i] == "--computable_params":
        args[i + 1] = f'"{args[i + 1]}"'  # Wrap the list in double quotes
        
    if args[i] == "--loadable_params":
        args[i + 1] = f'"{args[i + 1]}"'  # Wrap the list in double quotes

parser = argparse.ArgumentParser(description="Submit SLURM job with JSON configuration")
parser.add_argument("--folder", type=str, help="Custom folder path to use as the working directory", default="")
args_namespace = parser.parse_args()
working_directory = os.getcwd()



job_name = f"data_preprocessing"
all_conf_args = args
submit_slurm_job(script_path, all_conf_args, working_directory, job_name, submit)


Job submitted successfully: Submitted batch job 37162205


In [1]:
!rm -r slurm_sh/*.sh

In [ ]:
!rm -r slurm_outs/*.out

In [1]:
%%bash
squeue -u sokoi0a --format="%10i %9P %85j %8u %1t %4M %.6D  %10M %10l %8Q %7m"

JOBID      PARTITION NAME                                                                                  USER     S TIME  NODES  TIME       TIME_LIMIT PRIORITY MIN_MEM


In [ ]:
!tail -n 100 slurm_outs/data_preprocessing*.out

In [ ]:
!cat slurm_outs/data_preprocessing*.out

In [3]:
%%bash
scancel -u sokoi0a